<a href="https://colab.research.google.com/github/Yisihaq01/breast-histology-calibration/blob/main/Notebooks/03_breakhis_formal_development.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Formal Development-Only Model Selection

## CIS 631 — Breast Histopathology Calibration Research

This notebook implements the formal model-development stage of our breast histopathology calibration study.

The objective is to compare fine-tuned ResNet-18 and EfficientNet-B0 using the previously established patient-grouped three-fold Development partition.

### Experimental restrictions

- Only the **49 BreakHis Development patients** can be used for CNN model development and model selection.
- Only BreakHis **40× images** are used for primary model development.
- Existing patient assignments, Development folds, preprocessing parameters, and patch coordinates must remain unchanged.
- Calibration patients are reserved for fitting post-hoc calibrators after final CNN training.
- Test patients and external BACH/BRACS datasets are excluded from model development.
- Model selection will be based on Development-fold validation results.

The preprocessing parameters remain fixed:

| Parameter | Value |
|---|---|
| Source physical resolution | 0.49 µm/pixel |
| Target physical resolution | 0.50 µm/pixel |
| Patch size | 224 × 224 |
| Grid | 4 × 4 |
| Maximum patches per image | 16 |
| Optical density threshold | 0.08 |
| Saturation threshold | 0.05 |
| Minimum tissue fraction | 0.20 |

### Stage A: Data and fold verification

Before training, we reconstruct the complete Development patch manifest from previously saved patch manifests and verify that all three folds preserve patient-level separation.

No CNN training or held-out performance evaluation is performed during Stage A.


In [ ]:

# ============================================================
# Stage A — Environment and repository setup
# ============================================================

from pathlib import Path
import subprocess
import random

import numpy as np
import pandas as pd

from IPython.display import display

SEED = 631

random.seed(SEED)
np.random.seed(SEED)

REPO_ROOT = Path(
    "/content/breast-histology-calibration"
)

REPO_URL = (
    "https://github.com/Yisihaq01/"
    "breast-histology-calibration.git"
)

if not REPO_ROOT.exists():
    subprocess.run(
        [
            "git",
            "clone",
            REPO_URL,
            str(REPO_ROOT)
        ],
        check=True
    )
else:
    print(
        "Repository folder already exists."
    )

METADATA_DIR = REPO_ROOT / "Metadata"

RESULTS_DIR = (
    REPO_ROOT /
    "results" /
    "formal_development"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Repository:", REPO_ROOT)
print("Seed:", SEED)
print("Metadata directory:", METADATA_DIR)
print("Results directory:", RESULTS_DIR)


Repository: /content/breast-histology-calibration
Seed: 631
Metadata directory: /content/breast-histology-calibration/Metadata
Results directory: /content/breast-histology-calibration/results/formal_development


In [ ]:
# ============================================================
# Diagnose missing repository metadata
# ============================================================

from pathlib import Path
import subprocess

REPO_ROOT = Path(
    "/content/breast-histology-calibration"
)

def git_check(*args):
    result = subprocess.run(
        ["git", "-C", str(REPO_ROOT), *args],
        capture_output=True,
        text=True
    )
    print(result.stdout or result.stderr)

print("1. Current branch:")
git_check("branch", "--show-current")

print("\n2. Available remote branches:")
git_check("branch", "-r")

print("\n3. Latest commit:")
git_check("log", "-1", "--oneline")

print("\n4. Tracked metadata files:")
git_check("ls-files", "metadata")

print("\n5. Local metadata directory:")

metadata_dir = REPO_ROOT / "metadata"

if metadata_dir.exists():
    for path in sorted(metadata_dir.iterdir()):
        print(path.name)
else:
    print("Metadata directory does not exist.")

print("\n6. Repository status:")
git_check("status", "--short", "--branch")

1. Current branch:
main


2. Available remote branches:
  origin/Calibration_analysis
  origin/HEAD -> origin/main
  origin/main


3. Latest commit:
ae6f0d2 Merge branch 'main' of https://github.com/Yisihaq01/breast-histology-calibration


4. Tracked metadata files:


5. Local metadata directory:
Metadata directory does not exist.

6. Repository status:
## main...origin/main



In [ ]:
# ============================================================
# Check which GitHub branches contain the frozen metadata
# ============================================================

import subprocess
from pathlib import Path

REPO_ROOT = Path(
    "/content/breast-histology-calibration"
)

required_names = [
    "breakhis_40x_final_metadata.csv",
    "breakhis_patient_split.csv",
    "breakhis_development_folds.csv",
    "baseline_train_patch_manifest.csv",
    "baseline_val_patch_manifest.csv",
]

branches = [
    "origin/main",
    "origin/Calibration_analysis"
]

for branch in branches:

    result = subprocess.run(
        [
            "git",
            "-C", str(REPO_ROOT),
            "ls-tree",
            "-r",
            "--name-only",
            branch
        ],
        capture_output=True,
        text=True,
        check=True
    )

    tracked_files = result.stdout.splitlines()

    print(f"\n===== {branch} =====")

    for filename in required_names:

        matches = [
            path for path in tracked_files
            if Path(path).name == filename
        ]

        if matches:
            print("FOUND:", matches[0])
        else:
            print("MISSING:", filename)

# Check if Git is ignoring the metadata directory
print("\n===== Git ignore check =====")

result = subprocess.run(
    [
        "git",
        "-C", str(REPO_ROOT),
        "check-ignore",
        "-v",
        "--no-index",
        "metadata/breakhis_patient_split.csv"
    ],
    capture_output=True,
    text=True
)

print(
    result.stdout.strip()
    or "No matching ignore rule found."
)


===== origin/main =====
FOUND: Metadata/breakhis_40x_final_metadata.csv
FOUND: Metadata/breakhis_patient_split.csv
FOUND: Metadata/breakhis_development_folds.csv
FOUND: Metadata/baseline_train_patch_manifest.csv
FOUND: Metadata/baseline_val_patch_manifest.csv

===== origin/Calibration_analysis =====
FOUND: Metadata/breakhis_40x_final_metadata.csv
FOUND: Metadata/breakhis_patient_split.csv
FOUND: Metadata/breakhis_development_folds.csv
FOUND: Metadata/baseline_train_patch_manifest.csv
FOUND: Metadata/baseline_val_patch_manifest.csv

===== Git ignore check =====
No matching ignore rule found.


In [ ]:
# ============================================================
# Verify corrected GitHub metadata directory
# ============================================================

from pathlib import Path

REPO_ROOT = Path(
    "/content/breast-histology-calibration"
)

METADATA_DIR = REPO_ROOT / "Metadata"

print("Metadata directory:", METADATA_DIR)
print("Directory exists:", METADATA_DIR.exists())

required_files = {
    "final_40x_metadata":
        METADATA_DIR / "breakhis_40x_final_metadata.csv",

    "patient_split":
        METADATA_DIR / "breakhis_patient_split.csv",

    "development_folds":
        METADATA_DIR / "breakhis_development_folds.csv",

    "training_patch_manifest":
        METADATA_DIR / "baseline_train_patch_manifest.csv",

    "validation_patch_manifest":
        METADATA_DIR / "baseline_val_patch_manifest.csv"
}

for name, filepath in required_files.items():
    print(
        f"{name}:",
        "FOUND" if filepath.exists() else "MISSING"
    )

assert all(
    path.exists()
    for path in required_files.values()
)

print(
    "\nPASS: All five frozen metadata files located."
)

Metadata directory: /content/breast-histology-calibration/Metadata
Directory exists: True
final_40x_metadata: FOUND
patient_split: FOUND
development_folds: FOUND
training_patch_manifest: FOUND
validation_patch_manifest: FOUND

PASS: All five frozen metadata files located.


In [ ]:

# ============================================================
# Stage A — Inspect frozen metadata structure
# ============================================================

import pandas as pd

for name, filepath in required_files.items():

    df = pd.read_csv(filepath)

    print("\n" + "=" * 55)
    print(name.upper())
    print("=" * 55)

    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

    if name == "development_folds":
        print("\nFirst 5 fold assignments:")
        display(df.head(5))

    elif name == "patient_split":
        print("\nPatient split counts:")
        print(df["split"].value_counts())

    elif name in [
        "training_patch_manifest",
        "validation_patch_manifest"
    ]:
        print("\nFirst 2 patch records:")
        display(df.head(2))



FINAL_40X_METADATA
Rows: 1961
Columns: ['filename', 'relative_path', 'patient_id', 'diagnosis_code', 'diagnosis', 'original_subtype_code', 'original_subtype', 'subtype_code', 'subtype', 'magnification', 'image_number', 'split']

PATIENT_SPLIT
Rows: 81
Columns: ['patient_id', 'diagnosis_code', 'diagnosis', 'subtype_code', 'subtype', 'split']

Patient split counts:
split
development    49
calibration    16
test           16
Name: count, dtype: int64

DEVELOPMENT_FOLDS
Rows: 49
Columns: ['patient_id', 'diagnosis_code', 'diagnosis', 'subtype_code', 'subtype', 'split', 'development_fold']

First 5 fold assignments:


,patient_id,diagnosis_code,diagnosis,subtype_code,subtype,split,development_fold
0,14-29960CD,B,Benign,A,Adenosis,development,1
1,14-14134,B,Benign,F,Fibroadenoma,development,1
2,14-21998CD,B,Benign,F,Fibroadenoma,development,1
3,14-13200,B,Benign,TA,Tubular adenoma,development,1
4,14-3411F,B,Benign,TA,Tubular adenoma,development,1



TRAINING_PATCH_MANIFEST
Rows: 12034
Columns: ['patient_id', 'filename', 'relative_path', 'diagnosis', 'label', 'patch_index', 'x', 'y', 'tissue_fraction', 'used_fallback']

First 2 patch records:


,patient_id,filename,relative_path,diagnosis,label,patch_index,x,y,tissue_fraction,used_fallback
0,14-10147,SOB_M_MC-14-10147-40-009.png,BreaKHis_v1/BreaKHis_v1/histology_slides/breas...,Malignant,1,0,0,0,0.578823,False
1,14-10147,SOB_M_MC-14-10147-40-009.png,BreaKHis_v1/BreaKHis_v1/histology_slides/breas...,Malignant,1,1,154,0,0.518634,False



VALIDATION_PATCH_MANIFEST
Rows: 5861
Columns: ['patient_id', 'filename', 'relative_path', 'diagnosis', 'label', 'patch_index', 'x', 'y', 'tissue_fraction', 'used_fallback']

First 2 patch records:


,patient_id,filename,relative_path,diagnosis,label,patch_index,x,y,tissue_fraction,used_fallback
0,14-13200,SOB_B_TA-14-13200-40-012.png,BreaKHis_v1/BreaKHis_v1/histology_slides/breas...,Benign,0,0,0,0,0.928133,False
1,14-13200,SOB_B_TA-14-13200-40-012.png,BreaKHis_v1/BreaKHis_v1/histology_slides/breas...,Benign,0,1,154,0,0.830716,False


In [ ]:
# ============================================================
# Check which GitHub branches contain the frozen metadata
# ============================================================

import subprocess
from pathlib import Path

REPO_ROOT = Path(
    "/content/breast-histology-calibration"
)

required_names = [
    "breakhis_40x_final_metadata.csv",
    "breakhis_patient_split.csv",
    "breakhis_development_folds.csv",
    "baseline_train_patch_manifest.csv",
    "baseline_val_patch_manifest.csv",
]

branches = [
    "origin/main",
    "origin/Calibration_analysis"
]

for branch in branches:

    result = subprocess.run(
        [
            "git",
            "-C", str(REPO_ROOT),
            "ls-tree",
            "-r",
            "--name-only",
            branch
        ],
        capture_output=True,
        text=True,
        check=True
    )

    tracked_files = result.stdout.splitlines()

    print(f"\n===== {branch} =====")

    for filename in required_names:

        matches = [
            path for path in tracked_files
            if Path(path).name == filename
        ]

        if matches:
            print("FOUND:", matches[0])
        else:
            print("MISSING:", filename)

# Check if Git is ignoring the metadata directory
print("\n===== Git ignore check =====")

result = subprocess.run(
    [
        "git",
        "-C", str(REPO_ROOT),
        "check-ignore",
        "-v",
        "--no-index",
        "metadata/breakhis_patient_split.csv"
    ],
    capture_output=True,
    text=True
)

print(
    result.stdout.strip()
    or "No matching ignore rule found."
)


===== origin/main =====
FOUND: Metadata/breakhis_40x_final_metadata.csv
FOUND: Metadata/breakhis_patient_split.csv
FOUND: Metadata/breakhis_development_folds.csv
FOUND: Metadata/baseline_train_patch_manifest.csv
FOUND: Metadata/baseline_val_patch_manifest.csv

===== origin/Calibration_analysis =====
FOUND: Metadata/breakhis_40x_final_metadata.csv
FOUND: Metadata/breakhis_patient_split.csv
FOUND: Metadata/breakhis_development_folds.csv
FOUND: Metadata/baseline_train_patch_manifest.csv
FOUND: Metadata/baseline_val_patch_manifest.csv

===== Git ignore check =====
No matching ignore rule found.


In [ ]:

# ============================================================
# Load and verify frozen Development metadata
# ============================================================

import pandas as pd

image_metadata = pd.read_csv(
    required_files["final_40x_metadata"],
    dtype={"patient_id": str}
)

patient_split = pd.read_csv(
    required_files["patient_split"],
    dtype={"patient_id": str}
)

development_folds = pd.read_csv(
    required_files["development_folds"],
    dtype={"patient_id": str}
)

train_patches = pd.read_csv(
    required_files["training_patch_manifest"],
    dtype={"patient_id": str}
)

val_patches = pd.read_csv(
    required_files["validation_patch_manifest"],
    dtype={"patient_id": str}
)

# Standardize labels without changing stored files
for df in [image_metadata, patient_split, development_folds]:
    df["split"] = df["split"].str.strip().str.lower()

development_folds["development_fold"] = (
    development_folds["development_fold"].astype(int)
)

# Verify original data dimensions
assert len(image_metadata) == 1961
assert len(patient_split) == 81
assert len(development_folds) == 49

assert len(train_patches) == 12034
assert len(val_patches) == 5861

assert image_metadata["patient_id"].nunique() == 81
assert patient_split["patient_id"].nunique() == 81
assert development_folds["patient_id"].nunique() == 49

assert not image_metadata["filename"].duplicated().any()
assert not patient_split["patient_id"].duplicated().any()
assert not development_folds["patient_id"].duplicated().any()

assert image_metadata["magnification"].astype(str).eq("40").all()

# Frozen split membership
DEV_IDS = set(
    patient_split.loc[
        patient_split["split"] == "development",
        "patient_id"
    ]
)

CAL_IDS = set(
    patient_split.loc[
        patient_split["split"] == "calibration",
        "patient_id"
    ]
)

TEST_IDS = set(
    patient_split.loc[
        patient_split["split"] == "test",
        "patient_id"
    ]
)

assert len(DEV_IDS) == 49
assert len(CAL_IDS) == 16
assert len(TEST_IDS) == 16

assert DEV_IDS.isdisjoint(CAL_IDS)
assert DEV_IDS.isdisjoint(TEST_IDS)
assert CAL_IDS.isdisjoint(TEST_IDS)

assert (
    DEV_IDS | CAL_IDS | TEST_IDS
) == set(patient_split["patient_id"])

# Check image split agrees with authoritative patient split
split_lookup = patient_split.set_index("patient_id")["split"]

assert (
    image_metadata["split"]
    == image_metadata["patient_id"].map(split_lookup)
).all()

# Development subset
dev_images = image_metadata.loc[
    image_metadata["patient_id"].isin(DEV_IDS)
].copy()

assert len(dev_images) == 1133
assert dev_images["patient_id"].nunique() == 49

# Verify original Development folds
assert set(development_folds["patient_id"]) == DEV_IDS
assert development_folds["split"].eq("development").all()

fold_counts = (
    development_folds["development_fold"]
    .value_counts()
    .sort_index()
)

assert fold_counts.to_dict() == {
    1: 17,
    2: 16,
    3: 16
}

print("Development patients:", len(DEV_IDS))
print("Calibration patients:", len(CAL_IDS))
print("Test patients:", len(TEST_IDS))

print("\nDevelopment images:", len(dev_images))

print("\nDevelopment fold counts:")
display(fold_counts)

print("\nPASS: Frozen Development metadata verified.")


Development patients: 49
Calibration patients: 16
Test patients: 16

Development images: 1133

Development fold counts:


,count
development_fold,
1,17
2,16
3,16



PASS: Frozen Development metadata verified.


In [ ]:

# ============================================================
# Reconstruct Development patch manifest
# ============================================================

all_patches = pd.concat(
    [train_patches, val_patches],
    ignore_index=True
)

# Check total number of patches and membership
assert len(all_patches) == 17895

assert all_patches["patient_id"].nunique() == 49
assert all_patches["filename"].nunique() == 1133

assert set(all_patches["patient_id"]) == DEV_IDS
assert set(all_patches["filename"]) == set(dev_images["filename"])

# Check for repeated patch identities
assert not all_patches.duplicated(
    ["filename", "patch_index"]
).any()

# Build authoritative image lookup
image_lookup = dev_images.set_index("filename")

for column in [
    "patient_id",
    "diagnosis",
    "relative_path"
]:

    expected = all_patches["filename"].map(
        image_lookup[column]
    )

    assert expected.notna().all(), (
        f"Missing expected values for {column}"
    )

    assert (
        all_patches[column].astype(str).to_numpy()
        == expected.astype(str).to_numpy()
    ).all(), (
        f"Patch/image mismatch in {column}"
    )

# Verify binary labels
expected_labels = (
    all_patches["filename"]
    .map(image_lookup["diagnosis"])
    .eq("Malignant")
    .astype(int)
)

assert (
    all_patches["label"].astype(int).to_numpy()
    == expected_labels.to_numpy()
).all()

# Verify all patches use valid indices
assert all_patches["patch_index"].between(0, 15).all()
assert all_patches["tissue_fraction"].between(0, 1).all()

patches_per_image = (
    all_patches.groupby("filename").size()
)

assert patches_per_image.between(1, 16).all()

# Confirm that the old preliminary split used Fold 1
fold_lookup = development_folds.set_index(
    "patient_id"
)["development_fold"]

original_val_folds = set(
    val_patches["patient_id"].map(fold_lookup)
)

original_train_folds = set(
    train_patches["patient_id"].map(fold_lookup)
)

assert original_val_folds == {1}
assert original_train_folds == {2, 3}

assert set(train_patches["patient_id"]).isdisjoint(
    set(val_patches["patient_id"])
)

# Attach original frozen Development fold
all_patches["development_fold"] = (
    all_patches["patient_id"].map(fold_lookup)
)

assert all_patches["development_fold"].notna().all()

all_patches["development_fold"] = (
    all_patches["development_fold"].astype(int)
)

all_patches = all_patches.sort_values(
    [
        "development_fold",
        "patient_id",
        "filename",
        "patch_index"
    ]
).reset_index(drop=True)

print("Development patients:", all_patches["patient_id"].nunique())
print("Development images:", all_patches["filename"].nunique())
print("Development patches:", len(all_patches))

print("\nPatches per image:")
print(patches_per_image.describe())

print("\nPASS: Frozen Development patch manifest reconstructed.")


Development patients: 49
Development images: 1133
Development patches: 17895

Patches per image:
count    1133.000000
mean       15.794351
std         1.121223
min         1.000000
25%        16.000000
50%        16.000000
75%        16.000000
max        16.000000
dtype: float64

PASS: Frozen Development patch manifest reconstructed.


In [ ]:

# ============================================================
# Prepare patient-grouped three-fold CV
# ============================================================

cv_splits = {}
cv_summary_rows = []

for validation_fold in [1, 2, 3]:

    train_df = all_patches.loc[
        all_patches["development_fold"] != validation_fold
    ].copy()

    val_df = all_patches.loc[
        all_patches["development_fold"] == validation_fold
    ].copy()

    train_patients = set(train_df["patient_id"])
    val_patients = set(val_df["patient_id"])

    # No shared patients
    assert train_patients.isdisjoint(val_patients)

    # Together they cover all Development patients
    assert train_patients | val_patients == DEV_IDS

    # No held-out patients
    assert (
        train_patients | val_patients
    ).isdisjoint(CAL_IDS | TEST_IDS)

    # Save each fold in memory for later model training
    cv_splits[validation_fold] = {
        "train": train_df,
        "validation": val_df
    }

    cv_summary_rows.append({
        "validation_fold": validation_fold,
        "training_patients": len(train_patients),
        "validation_patients": len(val_patients),
        "training_images": train_df["filename"].nunique(),
        "validation_images": val_df["filename"].nunique(),
        "training_patches": len(train_df),
        "validation_patches": len(val_df)
    })

cv_summary = pd.DataFrame(cv_summary_rows)

display(cv_summary)

assert cv_summary["training_patients"].tolist() == [
    32, 33, 33
]

assert cv_summary["validation_patients"].tolist() == [
    17, 16, 16
]

assert cv_summary["validation_images"].sum() == 1133

assert cv_summary["validation_patches"].sum() == 17895

print(
    "\nPASS: All three patient-grouped Development "
    "folds are ready."
)


,validation_fold,training_patients,validation_patients,training_images,validation_images,training_patches,validation_patches
0,1,32,17,761,372,12034,5861
1,2,33,16,702,431,11085,6810
2,3,33,16,803,330,12671,5224



PASS: All three patient-grouped Development folds are ready.


In [ ]:

# ============================================================
# Save verified Development outputs
# ============================================================

from pathlib import Path

METADATA_DIR = REPO_ROOT / "Metadata"

RESULTS_DIR = (
    REPO_ROOT / "results" / "formal_development"
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

manifest_path = (
    METADATA_DIR /
    "breakhis_development_all_patch_manifest.csv"
)

summary_path = (
    RESULTS_DIR /
    "development_cv_preflight_summary.csv"
)

all_patches.to_csv(manifest_path, index=False)
cv_summary.to_csv(summary_path, index=False)

# Reopen files to verify they were written correctly
saved_manifest = pd.read_csv(
    manifest_path,
    dtype={"patient_id": str}
)

saved_summary = pd.read_csv(summary_path)

assert len(saved_manifest) == 17895
assert saved_manifest["patient_id"].nunique() == 49

assert len(saved_summary) == 3

print("Saved files:")
print(manifest_path)
print(summary_path)

print("\nPASS: Stage A completed successfully.")


Saved files:
/content/breast-histology-calibration/Metadata/breakhis_development_all_patch_manifest.csv
/content/breast-histology-calibration/results/formal_development/development_cv_preflight_summary.csv

PASS: Stage A completed successfully.


# Stage B — Fine-Tuned ResNet-18

## GPU and Dataset Verification

This stage implements formal ResNet-18 model development using only the frozen 49-patient BreakHis Development cohort.

All three original Development folds will be used for model selection. Calibration, Test, BACH, and BRACS remain excluded.

In [ ]:

# ============================================================
# Verify GPU and PyTorch environment
# ============================================================

import torch
import torchvision
import numpy as np
import pandas as pd

from pathlib import Path
from PIL import Image

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():

    device = torch.device("cuda")

    print("GPU:", torch.cuda.get_device_name(0))

    gpu_memory = (
        torch.cuda.get_device_properties(0).total_memory
        / (1024 ** 3)
    )

    print(f"GPU memory: {gpu_memory:.2f} GB")

else:
    device = torch.device("cpu")

    raise RuntimeError(
        "GPU is not available. "
        "Select T4 GPU in Colab before continuing."
    )

print("\nPASS: GPU environment is ready.")


PyTorch version: 2.11.0+cu130
Torchvision version: 0.26.0+cu130

CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB

PASS: GPU environment is ready.


In [ ]:

# ============================================================
# Load frozen Development manifest
# ============================================================

from pathlib import Path
import subprocess
import pandas as pd

REPO_ROOT = Path(
    "/content/breast-histology-calibration"
)

REPO_URL = (
    "https://github.com/Yisihaq01/"
    "breast-histology-calibration.git"
)

# Restore repository if the GPU runtime reset /content
if not REPO_ROOT.exists():

    subprocess.run(
        [
            "git",
            "clone",
            REPO_URL,
            str(REPO_ROOT)
        ],
        check=True
    )

METADATA_DIR = REPO_ROOT / "Metadata"

RESULTS_DIR = (
    REPO_ROOT /
    "results" /
    "formal_development"
)

manifest_path = (
    METADATA_DIR /
    "breakhis_development_all_patch_manifest.csv"
)

assert manifest_path.exists(), (
    "Frozen Development manifest is missing. "
    "Restore the committed Stage A file before continuing."
)

all_patches = pd.read_csv(
    manifest_path,
    dtype={"patient_id": str}
)

assert len(all_patches) == 17895

assert (
    all_patches["patient_id"].nunique() == 49
)

assert (
    all_patches["filename"].nunique() == 1133
)

assert set(
    all_patches["development_fold"].unique()
) == {1, 2, 3}

print("Repository:", REPO_ROOT)

print("\nDevelopment patients:",
      all_patches["patient_id"].nunique())

print("Development images:",
      all_patches["filename"].nunique())

print("Development patches:", len(all_patches))

print("\nPatients by fold:")

display(
    all_patches.groupby("development_fold")[
        "patient_id"
    ].nunique()
)

print(
    "\nPASS: Frozen Development manifest "
    "loaded successfully."
)


Repository: /content/breast-histology-calibration

Development patients: 49
Development images: 1133
Development patches: 17895

Patients by fold:


,patient_id
development_fold,
1,17
2,16
3,16



PASS: Frozen Development manifest loaded successfully.


In [ ]:

# ============================================================
# Locate original BreakHis dataset
# ============================================================

import importlib.util
import subprocess
import sys

if importlib.util.find_spec("kagglehub") is None:

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "kagglehub"
        ],
        check=True
    )

import kagglehub

# Reuse an existing dataset location when available
known_root = Path("/kaggle/input/breakhis")

if known_root.exists():

    DATASET_ROOT = known_root

else:

    DATASET_ROOT = Path(
        kagglehub.dataset_download(
            "ambarish/breakhis"
        )
    )

print("BreakHis dataset root:", DATASET_ROOT)
print("Dataset exists:", DATASET_ROOT.exists())

assert DATASET_ROOT.exists()

print("\nTop-level contents:")

for path in sorted(DATASET_ROOT.iterdir())[:15]:
    print("-", path.name)

print("\nPASS: BreakHis dataset located.")


Using Colab cache for faster access to the 'breakhis' dataset.
BreakHis dataset root: /kaggle/input/breakhis
Dataset exists: True

Top-level contents:
- BreaKHis_v1
- Folds.csv

PASS: BreakHis dataset located.


In [ ]:

# ============================================================
# Verify original image paths
# ============================================================

sample_rows = (
    all_patches[
        ["filename", "relative_path", "patient_id"]
    ]
    .drop_duplicates("filename")
    .sample(
        n=5,
        random_state=631
    )
)

print("Checking 5 frozen Development images:\n")

missing_images = []

for _, row in sample_rows.iterrows():

    image_path = (
        DATASET_ROOT /
        row["relative_path"]
    )

    if not image_path.exists():

        missing_images.append(str(image_path))

        print(
            "MISSING:",
            row["filename"]
        )

        continue

    with Image.open(image_path) as img:

        print(
            "FOUND:",
            row["filename"],
            "| Size:",
            img.size,
            "| Mode:",
            img.mode
        )

if missing_images:

    print("\nExample missing path:")
    print(missing_images[0])

    raise FileNotFoundError(
        "One or more frozen image paths could not be resolved. "
        "Check DATASET_ROOT before training."
    )

print(
    "\nPASS: Frozen Development image paths "
    "resolve correctly."
)


Checking 5 frozen Development images:

FOUND: SOB_M_DC-14-12312-40-007.png | Size: (700, 460) | Mode: RGB
FOUND: SOB_M_PC-14-15704-40-008.png | Size: (700, 460) | Mode: RGB
FOUND: SOB_M_MC-14-10147-40-004.png | Size: (700, 460) | Mode: RGB
FOUND: SOB_M_DC-14-13412-40-016.png | Size: (700, 460) | Mode: RGB
FOUND: SOB_M_DC-14-10926-40-004.png | Size: (700, 460) | Mode: RGB

PASS: Frozen Development image paths resolve correctly.


In [ ]:

# ============================================================
# Inspect original frozen preprocessing code
# ============================================================

import json
from pathlib import Path

# Find Notebook 01 in the cloned repository
notebook_candidates = [
    p for p in REPO_ROOT.rglob("*.ipynb")
    if "01_breakhis" in p.name.lower()
]

print("Matching notebooks:")

for path in notebook_candidates:
    print(path.relative_to(REPO_ROOT))

assert notebook_candidates, (
    "Notebook 01 was not found in the repository. "
    "Do not recreate preprocessing from memory."
)

original_notebook = notebook_candidates[0]

with open(
    original_notebook,
    "r",
    encoding="utf-8"
) as f:
    notebook_data = json.load(f)

keywords = [
    "0.49",
    "0.50",
    "lanczos",
    "resample",
    "physical_resolution",
    "target_mpp",
    "source_mpp",
    "resize_image",
    "patch_coordinates"
]

matching_cells = []

for index, cell in enumerate(
    notebook_data["cells"]
):

    if cell["cell_type"] != "code":
        continue

    source = "".join(
        cell.get("source", [])
    )

    source_lower = source.lower()

    if any(
        keyword in source_lower
        for keyword in keywords
    ):
        matching_cells.append(
            (index, source)
        )

print(
    "\nMatching preprocessing cells:",
    len(matching_cells)
)

for index, source in matching_cells[:8]:

    print("\n" + "=" * 65)
    print("NOTEBOOK 01 — CELL", index)
    print("=" * 65)

    print(source[:5000])


Matching notebooks:
Notebooks/01_breakhis_data_audit_and_split.ipynb
Notebooks/01_breakhis_audit.ipynb

Matching preprocessing cells: 8

NOTEBOOK 01 — CELL 57
print("Development images evaluated:",
      len(development_tissue))

print("\nTissue-fraction summary:")
print(
    development_tissue["tissue_fraction"]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

NOTEBOOK 01 — CELL 63
# ============================================================
# Frozen preprocessing parameters
# ============================================================

SOURCE_MPP_BREAKHIS = 0.49
TARGET_MPP = 0.50

PATCH_SIZE = 224
GRID_SIZE = 4
MAX_PATCHES = 16

MIN_PATCH_TISSUE_FRACTION = 0.20

OD_THRESHOLD = 0.08
SATURATION_THRESHOLD = 0.05

NOTEBOOK 01 — CELL 64
# ============================================================
# Physical-scale norma

In [ ]:

# ============================================================
# Audit all source paths and patch coordinates
# ============================================================

from PIL import Image
from collections import Counter

# One record per source image
unique_images = (
    all_patches[
        ["filename", "relative_path", "patient_id"]
    ]
    .drop_duplicates("filename")
    .copy()
)

assert len(unique_images) == 1133

missing_paths = []
image_sizes = Counter()

for row in unique_images.itertuples(index=False):

    image_path = (
        DATASET_ROOT / row.relative_path
    )

    if not image_path.is_file():
        missing_paths.append(str(image_path))
        continue

    with Image.open(image_path) as img:
        image_sizes[img.size] += 1

print("Development images checked:", len(unique_images))
print("Missing image paths:", len(missing_paths))

print("\nOriginal image dimensions:")

for size, count in sorted(image_sizes.items()):
    print(f"{size}: {count} images")

assert not missing_paths, (
    f"Missing source images: {missing_paths[:5]}"
)

# Verify saved patch coordinate integrity
assert all_patches[
    ["x", "y", "patch_index"]
].notna().all().all()

for column in ["x", "y", "patch_index"]:

    values = all_patches[column]

    assert (values >= 0).all()

    assert (
        values == values.astype(int)
    ).all()

assert all_patches["patch_index"].between(0, 15).all()

assert not all_patches.duplicated(
    ["filename", "patch_index"]
).any()

print("\nFrozen patch-coordinate ranges:")

print(
    "X:",
    all_patches["x"].min(),
    "to",
    all_patches["x"].max()
)

print(
    "Y:",
    all_patches["y"].min(),
    "to",
    all_patches["y"].max()
)

print(
    "Patch index:",
    all_patches["patch_index"].min(),
    "to",
    all_patches["patch_index"].max()
)

fallback_count = (
    all_patches["used_fallback"]
    .astype(str)
    .str.lower()
    .eq("true")
    .sum()
)

print(
    "\nFallback patches:",
    fallback_count
)

print(
    "\nPASS: All Development image paths "
    "and saved patch coordinates verified."
)


Development images checked: 1133
Missing image paths: 0

Original image dimensions:
(700, 456): 21 images
(700, 460): 1112 images

Frozen patch-coordinate ranges:
X: 0 to 462
Y: 0 to 227
Patch index: 0 to 15

Fallback patches: 0

PASS: All Development image paths and saved patch coordinates verified.


In [ ]:

# ============================================================
# Frozen preprocessing and transformations
# Reproduced from Notebook 01
# ============================================================

import random

import torch
from PIL import Image

from torchvision import transforms
from torchvision.transforms import functional as TF


# Frozen preprocessing parameters
SOURCE_MPP_BREAKHIS = 0.49
TARGET_MPP = 0.50

PATCH_SIZE = 224


def resample_to_target_mpp(
    image,
    source_mpp=SOURCE_MPP_BREAKHIS,
    target_mpp=TARGET_MPP
):
    """
    Resample the source image to 0.50 micrometers/pixel.
    Preserve the original Notebook 01 implementation.
    """

    image = image.convert("RGB")

    scale = source_mpp / target_mpp

    new_width = round(image.width * scale)
    new_height = round(image.height * scale)

    return image.resize(
        (new_width, new_height),
        Image.Resampling.LANCZOS
    )


class RandomRightAngleRotation:

    def __call__(self, image):

        angle = random.choice(
            [0, 90, 180, 270]
        )

        return TF.rotate(
            image,
            angle
        )


# Training: geometric augmentation only
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    RandomRightAngleRotation(),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# Validation: no random augmentation
val_transform = transforms.Compose([
    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# Verify original physical-resolution behavior
example_size = (700, 460)

example_image = Image.new(
    "RGB",
    example_size
)

resampled_example = resample_to_target_mpp(
    example_image
)

assert resampled_example.size == (686, 451)

print("Original image:", example_size)
print("Resampled image:", resampled_example.size)

print("\nTraining transformations:")
print(train_transform)

print("\nValidation transformations:")
print(val_transform)

print(
    "\nPASS: Frozen preprocessing "
    "and transformations restored."
)


Original image: (700, 460)
Resampled image: (686, 451)

Training transformations:
Compose(
    RandomHorizontalFlip(p=0.5)
    RandomVerticalFlip(p=0.5)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)

Validation transformations:
Compose(
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)

PASS: Frozen preprocessing and transformations restored.


In [ ]:

# ============================================================
# BreakHis patch Dataset
# ============================================================

from pathlib import Path

from torch.utils.data import Dataset, DataLoader


class BreakHisPatchDataset(Dataset):

    def __init__(
        self,
        manifest,
        dataset_root,
        transform=None
    ):

        self.manifest = (
            manifest
            .reset_index(drop=True)
            .copy()
        )

        self.dataset_root = Path(
            dataset_root
        )

        self.transform = transform

        required_columns = {
            "relative_path",
            "patient_id",
            "filename",
            "label",
            "patch_index",
            "x",
            "y"
        }

        missing_columns = (
            required_columns
            - set(self.manifest.columns)
        )

        if missing_columns:
            raise ValueError(
                f"Missing manifest columns: "
                f"{missing_columns}"
            )

        if not set(
            self.manifest["label"].unique()
        ).issubset({0, 1}):
            raise ValueError(
                "Dataset labels must be binary."
            )

    def __len__(self):

        return len(self.manifest)

    def __getitem__(self, index):

        row = self.manifest.iloc[index]

        image_path = (
            self.dataset_root /
            row["relative_path"]
        )

        # Load original-resolution source image
        with Image.open(image_path) as source:
            image = resample_to_target_mpp(
                source
            )

        # Use frozen patch coordinates
        x = int(row["x"])
        y = int(row["y"])

        # Prevent accidental padded/out-of-bounds crops
        if (
            x < 0
            or y < 0
            or x + PATCH_SIZE > image.width
            or y + PATCH_SIZE > image.height
        ):
            raise ValueError(
                "Invalid frozen crop coordinates: "
                f"{row['filename']}, x={x}, y={y}, "
                f"resampled_size={image.size}"
            )

        patch = image.crop(
            (
                x,
                y,
                x + PATCH_SIZE,
                y + PATCH_SIZE
            )
        )

        if self.transform is not None:
            patch = self.transform(patch)

        label = torch.tensor(
            float(row["label"]),
            dtype=torch.float32
        )

        return {
            "image": patch,
            "label": label,
            "patient_id": str(row["patient_id"]),
            "filename": str(row["filename"]),
            "patch_index": int(row["patch_index"])
        }


print("PASS: BreakHisPatchDataset defined.")


PASS: BreakHisPatchDataset defined.


In [ ]:

# ============================================================
# Prepare Fold 1 DataLoaders
# ============================================================

import numpy as np
import random
import torch

from torch.utils.data import DataLoader


VALIDATION_FOLD = 1
BATCH_SIZE = 32
NUM_WORKERS = 2

# Frozen patient-grouped assignments
fold_train_df = (
    all_patches[
        all_patches["development_fold"]
        != VALIDATION_FOLD
    ]
    .copy()
)

fold_val_df = (
    all_patches[
        all_patches["development_fold"]
        == VALIDATION_FOLD
    ]
    .copy()
)

# Patient-level leakage checks
train_patient_ids = set(
    fold_train_df["patient_id"]
)

val_patient_ids = set(
    fold_val_df["patient_id"]
)

assert train_patient_ids.isdisjoint(
    val_patient_ids
)

assert len(train_patient_ids) == 32
assert len(val_patient_ids) == 17

assert len(fold_train_df) == 12034
assert len(fold_val_df) == 5861


# Construct datasets
train_dataset = BreakHisPatchDataset(
    manifest=fold_train_df,
    dataset_root=DATASET_ROOT,
    transform=train_transform
)

val_dataset = BreakHisPatchDataset(
    manifest=fold_val_df,
    dataset_root=DATASET_ROOT,
    transform=val_transform
)


# Seed independent DataLoader worker processes
def seed_worker(worker_id):

    worker_seed = (
        torch.initial_seed() % (2 ** 32)
    )

    np.random.seed(worker_seed)
    random.seed(worker_seed)


loader_generator = torch.Generator()
loader_generator.manual_seed(631)


# Training DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    worker_init_fn=seed_worker,
    generator=loader_generator
)


# Validation DataLoader
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)


print("Fold:", VALIDATION_FOLD)

print(
    "Training patients:",
    len(train_patient_ids)
)

print(
    "Validation patients:",
    len(val_patient_ids)
)

print(
    "\nTraining patches:",
    len(train_dataset)
)

print(
    "Validation patches:",
    len(val_dataset)
)

print(
    "\nTraining batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(val_loader)
)

print(
    "\nPASS: Fold 1 DataLoaders constructed."
)


Fold: 1
Training patients: 32
Validation patients: 17

Training patches: 12034
Validation patches: 5861

Training batches: 377
Validation batches: 184

PASS: Fold 1 DataLoaders constructed.


In [ ]:

# ============================================================
# Test training and validation batches
# ============================================================

train_batch = next(
    iter(train_loader)
)

val_batch = next(
    iter(val_loader)
)


def verify_batch(batch, name):

    images = batch["image"]
    labels = batch["label"]

    print(f"\n{name} batch:")

    print("Image tensor:", images.shape)
    print("Label tensor:", labels.shape)

    print(
        "Distinct patients:",
        len(set(batch["patient_id"]))
    )

    print(
        "Distinct source images:",
        len(set(batch["filename"]))
    )

    print(
        "Labels:",
        torch.unique(labels).tolist()
    )

    # N x 3 x 224 x 224
    assert images.ndim == 4
    assert images.shape[1:] == (3, 224, 224)

    assert labels.ndim == 1
    assert labels.shape[0] == images.shape[0]

    assert torch.isfinite(images).all()
    assert torch.isfinite(labels).all()

    assert set(
        torch.unique(labels).tolist()
    ).issubset({0.0, 1.0})

    assert len(batch["patient_id"]) == len(labels)
    assert len(batch["filename"]) == len(labels)

    print(f"PASS: {name} batch verified.")


verify_batch(
    train_batch,
    "Training"
)

verify_batch(
    val_batch,
    "Validation"
)


# Transfer one batch to GPU
gpu_images = train_batch["image"].to(
    device,
    non_blocking=True
)

gpu_labels = train_batch["label"].to(
    device,
    non_blocking=True
)

assert gpu_images.is_cuda
assert gpu_labels.is_cuda

print("\nGPU tensor shape:", gpu_images.shape)

print(
    "GPU device:",
    gpu_images.device
)

print(
    "\nPASS: Training and validation "
    "DataLoaders are ready for ResNet-18."
)



Training batch:
Image tensor: torch.Size([32, 3, 224, 224])
Label tensor: torch.Size([32])
Distinct patients: 20
Distinct source images: 32
Labels: [0.0, 1.0]
PASS: Training batch verified.

Validation batch:
Image tensor: torch.Size([32, 3, 224, 224])
Label tensor: torch.Size([32])
Distinct patients: 1
Distinct source images: 2
Labels: [0.0]
PASS: Validation batch verified.

GPU tensor shape: torch.Size([32, 3, 224, 224])
GPU device: cuda:0

PASS: Training and validation DataLoaders are ready for ResNet-18.


In [ ]:

# ============================================================
# Formal ResNet-18 training configuration
# ============================================================

import random
import numpy as np
import torch

SEED = 631

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_all_seeds(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

TRAIN_CONFIG = {
    "architecture": "resnet18",
    "pretrained_weights": "IMAGENET1K_V1",
    "finetune_layers": ["layer4", "fc"],
    "freeze_batchnorm_statistics": True,
    "batch_size": 32,
    "epochs": 5,
    "lr_layer4": 1e-4,
    "lr_fc": 5e-4,
    "weight_decay": 1e-4,
    "seed": 631,
    "selection_metric": "image_auroc",
    "validation_folds": [1, 2, 3]
}

print("Formal training configuration:\n")

for key, value in TRAIN_CONFIG.items():
    print(f"{key}: {value}")

print("\nPASS: Training configuration defined.")


Formal training configuration:

architecture: resnet18
pretrained_weights: IMAGENET1K_V1
finetune_layers: ['layer4', 'fc']
freeze_batchnorm_statistics: True
batch_size: 32
epochs: 5
lr_layer4: 0.0001
lr_fc: 0.0005
weight_decay: 0.0001
seed: 631
selection_metric: image_auroc
validation_folds: [1, 2, 3]

PASS: Training configuration defined.


In [ ]:

# ============================================================
# Build fine-tuned ResNet-18
# ============================================================

import torch.nn as nn

from torchvision.models import (
    resnet18,
    ResNet18_Weights
)


def build_resnet18_finetuned(seed=631):

    set_all_seeds(seed)

    model = resnet18(
        weights=ResNet18_Weights.IMAGENET1K_V1
    )

    # Freeze the entire pretrained backbone initially
    for parameter in model.parameters():
        parameter.requires_grad = False

    # Fine-tune the last residual block
    for parameter in model.layer4.parameters():
        parameter.requires_grad = True

    # Replace the ImageNet classification layer
    in_features = model.fc.in_features

    model.fc = nn.Linear(
        in_features,
        1
    )

    return model


def set_finetuning_train_mode(model):
    """
    Enable training while preventing BatchNorm running
    statistics from updating.
    """

    model.train()

    for module in model.modules():
        if isinstance(
            module,
            nn.modules.batchnorm._BatchNorm
        ):
            module.eval()


model = build_resnet18_finetuned(
    seed=SEED
).to(device)

set_finetuning_train_mode(model)

total_parameters = sum(
    p.numel()
    for p in model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

frozen_parameters = (
    total_parameters - trainable_parameters
)

print("Total parameters:", f"{total_parameters:,}")
print("Trainable parameters:", f"{trainable_parameters:,}")
print("Frozen parameters:", f"{frozen_parameters:,}")

print(
    "\nTrainable percentage:",
    f"{100 * trainable_parameters / total_parameters:.2f}%"
)

assert trainable_parameters > 513
assert trainable_parameters < total_parameters

assert model.fc.out_features == 1

print("\nPASS: Fine-tuned ResNet-18 initialized.")


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 171MB/s]


Total parameters: 11,177,025
Trainable parameters: 8,394,241
Frozen parameters: 2,782,784

Trainable percentage: 75.10%

PASS: Fine-tuned ResNet-18 initialized.


In [ ]:

# ============================================================
# Loss function and optimizer
# ============================================================

train_label_counts = (
    fold_train_df["label"]
    .astype(int)
    .value_counts()
)

n_benign = int(train_label_counts.get(0, 0))
n_malignant = int(train_label_counts.get(1, 0))

assert n_benign > 0
assert n_malignant > 0

pos_weight_value = n_benign / n_malignant

pos_weight = torch.tensor(
    [pos_weight_value],
    dtype=torch.float32,
    device=device
)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

optimizer = torch.optim.AdamW(
    [
        {
            "params": model.layer4.parameters(),
            "lr": TRAIN_CONFIG["lr_layer4"]
        },
        {
            "params": model.fc.parameters(),
            "lr": TRAIN_CONFIG["lr_fc"]
        }
    ],
    weight_decay=TRAIN_CONFIG["weight_decay"]
)

print("Fold 1 training class counts:")
print("Benign patches:", n_benign)
print("Malignant patches:", n_malignant)

print(
    "\nPositive class weight:",
    round(pos_weight_value, 6)
)

print("\nOptimizer:", type(optimizer).__name__)
print("Layer4 LR:", optimizer.param_groups[0]["lr"])
print("FC LR:", optimizer.param_groups[1]["lr"])

print("\nPASS: Training loss and optimizer configured.")


Fold 1 training class counts:
Benign patches: 3301
Malignant patches: 8733

Positive class weight: 0.377992

Optimizer: AdamW
Layer4 LR: 0.0001
FC LR: 0.0005

PASS: Training loss and optimizer configured.


In [ ]:

# ============================================================
# ResNet-18 forward/backward smoke test
# ============================================================

import torch.nn as nn

set_finetuning_train_mode(model)

batch = next(iter(train_loader))

images = batch["image"].to(
    device,
    non_blocking=True
)

labels = batch["label"].to(
    device,
    non_blocking=True
)

# Save BatchNorm running statistics before the test
bn_before = {
    name: module.running_mean.detach().clone()
    for name, module in model.named_modules()
    if isinstance(module, nn.BatchNorm2d)
}

optimizer.zero_grad(set_to_none=True)

# Mixed precision reduces GPU memory use
with torch.autocast(
    device_type="cuda",
    dtype=torch.float16
):
    logits = model(images).flatten()
    loss = criterion(logits, labels)

assert logits.shape == labels.shape
assert torch.isfinite(logits).all()
assert torch.isfinite(loss)

# Backpropagate, without optimizer.step()
loss.backward()

# Check trainable parameters received gradients
layer4_has_grad = any(
    p.grad is not None
    for p in model.layer4.parameters()
    if p.requires_grad
)

fc_has_grad = any(
    p.grad is not None
    for p in model.fc.parameters()
    if p.requires_grad
)

assert layer4_has_grad
assert fc_has_grad

# Check frozen parameters did not receive gradients
frozen_grads = [
    name
    for name, parameter in model.named_parameters()
    if not parameter.requires_grad
    and parameter.grad is not None
]

assert not frozen_grads

# Check BatchNorm statistics have not changed
for name, module in model.named_modules():
    if isinstance(module, nn.BatchNorm2d):
        assert torch.equal(
            bn_before[name],
            module.running_mean
        )

print("Input shape:", images.shape)
print("Output logits shape:", logits.shape)
print("Loss:", float(loss.item()))

print("\nLayer4 gradients:", layer4_has_grad)
print("FC gradients:", fc_has_grad)
print("Frozen-layer gradients:", len(frozen_grads))

print(
    "BatchNorm running statistics:",
    "unchanged"
)

optimizer.zero_grad(set_to_none=True)

del batch, images, labels, logits, loss

print(
    "\nPASS: ResNet-18 forward/backward smoke test."
)


Input shape: torch.Size([32, 3, 224, 224])
Output logits shape: torch.Size([32])
Loss: 0.3471906781196594

Layer4 gradients: True
FC gradients: True
Frozen-layer gradients: 0
BatchNorm running statistics: unchanged

PASS: ResNet-18 forward/backward smoke test.


In [ ]:

# ============================================================
# Image-level validation metrics
# ============================================================

import numpy as np
import pandas as pd

from scipy.special import expit
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    log_loss
)


def calculate_ece(y_true, probabilities, n_bins=10):
    """
    Equal-width expected calibration error for
    predicted probability of malignancy.
    """

    y_true = np.asarray(y_true, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)

    edges = np.linspace(0.0, 1.0, n_bins + 1)

    bin_ids = np.searchsorted(
        edges, probabilities, side="right"
    ) - 1

    bin_ids = np.clip(bin_ids, 0, n_bins - 1)

    ece = 0.0

    for bin_id in range(n_bins):

        mask = bin_ids == bin_id

        if not np.any(mask):
            continue

        observed = np.mean(y_true[mask])
        predicted = np.mean(probabilities[mask])

        ece += (
            np.mean(mask) *
            abs(observed - predicted)
        )

    return float(ece)


def calculate_image_metrics(y_true, probabilities):

    y_true = np.asarray(y_true, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)

    assert set(np.unique(y_true)) == {0, 1}
    assert np.isfinite(probabilities).all()
    assert np.all((probabilities >= 0) & (probabilities <= 1))

    predicted_labels = (probabilities >= 0.5).astype(int)

    probabilities_safe = np.clip(
        probabilities, 1e-7, 1 - 1e-7
    )

    return {
        "auroc": float(roc_auc_score(y_true, probabilities)),
        "auprc": float(average_precision_score(
            y_true, probabilities
        )),
        "balanced_accuracy": float(balanced_accuracy_score(
            y_true, predicted_labels
        )),
        "ece_10": calculate_ece(
            y_true, probabilities, n_bins=10
        ),
        "brier": float(brier_score_loss(
            y_true, probabilities
        )),
        "nll": float(log_loss(
            y_true, probabilities_safe, labels=[0, 1]
        ))
    }


def aggregate_patch_predictions(patch_predictions):
    """
    Aggregate patch-level logits into source-image
    predictions without treating patches as independent.
    """

    assert not patch_predictions.duplicated(
        ["filename", "patch_index"]
    ).any()

    assert (
        patch_predictions.groupby("filename")["label"]
        .nunique().max() == 1
    )

    assert (
        patch_predictions.groupby("filename")["patient_id"]
        .nunique().max() == 1
    )

    patch_predictions = patch_predictions.copy()

    patch_predictions["patch_probability"] = expit(
        patch_predictions["patch_logit"].to_numpy()
    )

    image_predictions = (
        patch_predictions.groupby("filename", as_index=False)
        .agg(
            patient_id=("patient_id", "first"),
            label=("label", "first"),
            image_logit=("patch_logit", "mean"),
            mean_patch_probability=("patch_probability", "mean"),
            n_patches=("patch_index", "size")
        )
    )

    # Primary frozen aggregation
    image_predictions["probability"] = expit(
        image_predictions["image_logit"].to_numpy()
    )

    image_predictions["prediction"] = (
        image_predictions["probability"] >= 0.5
    ).astype(int)

    return image_predictions


print("PASS: Image-level aggregation and metrics defined.")


PASS: Image-level aggregation and metrics defined.


In [ ]:

# ============================================================
# Training and validation functions
# ============================================================

import torch
from tqdm.auto import tqdm


def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler,
    device
):

    set_finetuning_train_mode(model)

    total_loss = 0.0
    total_examples = 0

    progress = tqdm(
        loader,
        desc="Training",
        leave=False
    )

    for batch in progress:

        images = batch["image"].to(
            device, non_blocking=True
        )

        labels = batch["label"].to(
            device, non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            logits = model(images).flatten()
            loss = criterion(logits, labels)

        if not torch.isfinite(loss):
            raise RuntimeError(
                "Non-finite training loss encountered."
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_size = len(labels)

        total_loss += float(loss.item()) * batch_size
        total_examples += batch_size

        progress.set_postfix(
            loss=f"{total_loss / total_examples:.4f}"
        )

    return total_loss / total_examples


@torch.inference_mode()
def validate_one_epoch(model, loader, device):

    model.eval()

    rows = []

    for batch in tqdm(
        loader,
        desc="Validation",
        leave=False
    ):

        images = batch["image"].to(
            device, non_blocking=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            logits = model(images).flatten()

        logits_cpu = (
            logits.float().cpu().numpy()
        )

        labels_cpu = (
            batch["label"].numpy()
        )

        patch_indices = (
            batch["patch_index"].tolist()
        )

        for i in range(len(logits_cpu)):

            rows.append({
                "filename": batch["filename"][i],
                "patient_id": batch["patient_id"][i],
                "label": int(labels_cpu[i]),
                "patch_index": int(patch_indices[i]),
                "patch_logit": float(logits_cpu[i])
            })

    patch_predictions = pd.DataFrame(rows)

    image_predictions = aggregate_patch_predictions(
        patch_predictions
    )

    metrics = calculate_image_metrics(
        image_predictions["label"],
        image_predictions["probability"]
    )

    metrics["n_images"] = int(len(image_predictions))
    metrics["n_patients"] = int(
        image_predictions["patient_id"].nunique()
    )

    return metrics, patch_predictions, image_predictions


print("PASS: Training and validation functions defined.")


PASS: Training and validation functions defined.


In [ ]:

# ============================================================
# Formal ResNet-18 Fold 1 training
# ============================================================

import gc
import json
from pathlib import Path

import torch
import torch.nn as nn


FOLD = 1
FOLD_SEED = SEED + FOLD

FOLD_RESULTS_DIR = (
    RESULTS_DIR / "resnet18" / "fold_1"
)

FOLD_RESULTS_DIR.mkdir(
    parents=True, exist_ok=True
)

CHECKPOINT_DIR = Path(
    "/content/cis631_model_checkpoints"
)

CHECKPOINT_DIR.mkdir(
    parents=True, exist_ok=True
)

checkpoint_path = (
    CHECKPOINT_DIR / "resnet18_fold1_best.pt"
)

history_path = (
    FOLD_RESULTS_DIR / "training_history.csv"
)

patch_predictions_path = (
    FOLD_RESULTS_DIR / "best_validation_patch_predictions.csv"
)

image_predictions_path = (
    FOLD_RESULTS_DIR / "best_validation_image_predictions.csv"
)

summary_path = (
    FOLD_RESULTS_DIR / "best_validation_summary.json"
)


# Reinitialize everything for the formal Fold 1 run.
# The earlier smoke test did not update weights, but we
# still start with a fresh model for reproducibility.

del model, optimizer, criterion
gc.collect()
torch.cuda.empty_cache()

set_all_seeds(FOLD_SEED)

model = build_resnet18_finetuned(
    seed=FOLD_SEED
).to(device)

train_loader.generator.manual_seed(FOLD_SEED)


# Training-only class weighting
n_benign = int(
    (fold_train_df["label"] == 0).sum()
)

n_malignant = int(
    (fold_train_df["label"] == 1).sum()
)

pos_weight_value = n_benign / n_malignant

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        [pos_weight_value],
        dtype=torch.float32,
        device=device
    )
)

optimizer = torch.optim.AdamW(
    [
        {
            "params": model.layer4.parameters(),
            "lr": TRAIN_CONFIG["lr_layer4"]
        },
        {
            "params": model.fc.parameters(),
            "lr": TRAIN_CONFIG["lr_fc"]
        }
    ],
    weight_decay=TRAIN_CONFIG["weight_decay"]
)

scaler = torch.amp.GradScaler("cuda")


# Verify validation cohort before training
assert len(fold_val_df) == 5861
assert fold_val_df["filename"].nunique() == 372
assert fold_val_df["patient_id"].nunique() == 17

best_auc = -float("inf")
best_epoch = None
history_rows = []


print("=" * 60)
print("FORMAL RESNET-18 — DEVELOPMENT FOLD 1")
print("=" * 60)
print("Training patients:", 32)
print("Validation patients:", 17)
print("Epochs:", TRAIN_CONFIG["epochs"])
print("Positive class weight:", round(pos_weight_value, 6))
print("Device:", device)
print("=" * 60)


for epoch in range(1, TRAIN_CONFIG["epochs"] + 1):

    print(
        f"\nEpoch {epoch}/{TRAIN_CONFIG['epochs']}"
    )

    training_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        scaler=scaler,
        device=device
    )

    (
        validation_metrics,
        patch_predictions,
        image_predictions
    ) = validate_one_epoch(
        model=model,
        loader=val_loader,
        device=device
    )

    assert validation_metrics["n_images"] == 372
    assert validation_metrics["n_patients"] == 17
    assert len(patch_predictions) == 5861

    epoch_result = {
        "epoch": epoch,
        "training_loss": training_loss,
        **validation_metrics
    }

    history_rows.append(epoch_result)

    print(f"Training loss: {training_loss:.4f}")
    print(f"Image AUROC:   {validation_metrics['auroc']:.4f}")
    print(f"Image AUPRC:   {validation_metrics['auprc']:.4f}")
    print(f"Balanced acc.: {validation_metrics['balanced_accuracy']:.4f}")
    print(f"ECE (10 bins): {validation_metrics['ece_10']:.4f}")
    print(f"Brier score:   {validation_metrics['brier']:.4f}")
    print(f"NLL:           {validation_metrics['nll']:.4f}")

    # Selection criterion: image-level AUROC.
    # Strict > keeps the earlier epoch in an exact tie.
    if validation_metrics["auroc"] > best_auc:

        best_auc = validation_metrics["auroc"]
        best_epoch = epoch

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "epoch": epoch,
                "validation_fold": FOLD,
                "seed": FOLD_SEED,
                "training_config": TRAIN_CONFIG,
                "validation_metrics": validation_metrics
            },
            checkpoint_path
        )

        patch_predictions.to_csv(
            patch_predictions_path,
            index=False
        )

        image_predictions.to_csv(
            image_predictions_path,
            index=False
        )

        print(
            f"NEW BEST CHECKPOINT — Epoch {epoch}"
        )

    pd.DataFrame(history_rows).to_csv(
        history_path,
        index=False
    )


best_history_row = next(
    row for row in history_rows
    if row["epoch"] == best_epoch
)

summary = {
    "architecture": "resnet18",
    "validation_fold": FOLD,
    "best_epoch": best_epoch,
    "best_validation_auroc": best_auc,
    "n_training_patients": 32,
    "n_validation_patients": 17,
    "n_validation_images": 372,
    "selection_metric": "image_auroc",
    "checkpoint_saved_separately": str(checkpoint_path),
    "training_config": TRAIN_CONFIG,
    "best_epoch_metrics": best_history_row
}

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)


print("\n" + "=" * 60)
print("FOLD 1 TRAINING COMPLETE")
print("=" * 60)
print("Best epoch:", best_epoch)
print(f"Best AUROC: {best_auc:.6f}")

print("\nSaved:")
print(history_path)
print(patch_predictions_path)
print(image_predictions_path)
print(summary_path)
print(checkpoint_path)

print("\nPASS: Formal ResNet-18 Fold 1 completed.")


FORMAL RESNET-18 — DEVELOPMENT FOLD 1
Training patients: 32
Validation patients: 17
Epochs: 5
Positive class weight: 0.377992
Device: cuda

Epoch 1/5


Training:   0%|          | 0/377 [00:00<?, ?it/s]

Validation:   0%|          | 0/184 [00:00<?, ?it/s]

Training loss: 0.0757
Image AUROC:   0.9857
Image AUPRC:   0.9934
Balanced acc.: 0.9612
ECE (10 bins): 0.0423
Brier score:   0.0399
NLL:           0.1432
NEW BEST CHECKPOINT — Epoch 1

Epoch 2/5


Training:   0%|          | 0/377 [00:00<?, ?it/s]

...

Training loss: 0.0159
Image AUROC:   0.9890
Image AUPRC:   0.9949
Balanced acc.: 0.9432
ECE (10 bins): 0.0428
Brier score:   0.0420
NLL:           0.1501

Epoch 5/5


Training:   0%|          | 0/377 [00:00<?, ?it/s]

Validation:   0%|          | 0/184 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7de5a9194d60>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7de5a9194d60>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

Training loss: 0.0103
Image AUROC:   0.9925
Image AUPRC:   0.9967
Balanced acc.: 0.9405
ECE (10 bins): 0.0747
Brier score:   0.0632
NLL:           0.2568
NEW BEST CHECKPOINT — Epoch 5

FOLD 1 TRAINING COMPLETE
Best epoch: 5
Best AUROC: 0.992493

Saved:
/content/breast-histology-calibration/results/formal_development/resnet18/fold_1/training_history.csv
/content/breast-histology-calibration/results/formal_development/resnet18/fold_1/best_validation_patch_predictions.csv
/content/breast-histology-calibration/results/formal_development/resnet18/fold_1/best_validation_image_predictions.csv
/content/breast-histology-calibration/results/formal_development/resnet18/fold_1/best_validation_summary.json
/content/cis631_model_checkpoints/resnet18_fold1_best.pt

PASS: Formal ResNet-18 Fold 1 completed.


In [ ]:

# ============================================================
# Inspect Fold 1 training results
# ============================================================

history_df = pd.read_csv(history_path)

display(
    history_df[
        [
            "epoch",
            "training_loss",
            "auroc",
            "auprc",
            "balanced_accuracy",
            "ece_10",
            "brier",
            "nll"
        ]
    ]
)

print("Best epoch:", best_epoch)
print("Best image-level AUROC:", round(best_auc, 6))

best_image_predictions = pd.read_csv(
    image_predictions_path
)

best_patch_predictions = pd.read_csv(
    patch_predictions_path
)

assert len(best_image_predictions) == 372
assert len(best_patch_predictions) == 5861

assert best_image_predictions["patient_id"].nunique() == 17

assert checkpoint_path.exists()

print("\nBest checkpoint size (MB):",
      round(checkpoint_path.stat().st_size / (1024**2), 2))

print("\nPASS: Fold 1 outputs verified.")


,epoch,training_loss,auroc,auprc,balanced_accuracy,ece_10,brier,nll
0,1,0.075701,0.985651,0.993445,0.961205,0.042321,0.039914,0.143225
1,2,0.024872,0.991962,0.996049,0.969359,0.021038,0.026373,0.101626
2,3,0.020577,0.986681,0.993847,0.823862,0.220611,0.187349,0.937155
3,4,0.015884,0.989039,0.994938,0.943169,0.042784,0.041961,0.150082
4,5,0.010348,0.992493,0.996657,0.940462,0.074696,0.063208,0.256791


Best epoch: 5
Best image-level AUROC: 0.992493

Best checkpoint size (MB): 42.71

PASS: Fold 1 outputs verified.


In [ ]:

# ============================================================
# Verify saved Fold 1 artifacts
# ============================================================

import json
import hashlib
import numpy as np
import pandas as pd
import torch

# Confirm every file exists
artifact_paths = [
    history_path,
    patch_predictions_path,
    image_predictions_path,
    summary_path,
    checkpoint_path
]

for path in artifact_paths:
    assert path.is_file(), f"Missing artifact: {path}"
    print(
        path.name,
        "|",
        round(path.stat().st_size / (1024 ** 2), 3),
        "MB"
    )

# Reopen saved results
saved_history = pd.read_csv(history_path)

saved_patches = pd.read_csv(
    patch_predictions_path,
    dtype={"patient_id": str}
)

saved_images = pd.read_csv(
    image_predictions_path,
    dtype={"patient_id": str}
)

with open(summary_path, "r") as f:
    saved_summary = json.load(f)

# Verify expected patient and image counts
assert len(saved_history) == 5
assert len(saved_patches) == 5861
assert len(saved_images) == 372
assert saved_images["patient_id"].nunique() == 17

# Verify patient membership
assert set(saved_images["patient_id"]) == val_patient_ids

# Verify checkpoint selection
assert saved_summary["best_epoch"] == 5
assert saved_summary["validation_fold"] == 1

selected_row = saved_history.loc[
    saved_history["epoch"] == 5
].iloc[0]

assert np.isclose(
    selected_row["auroc"],
    saved_summary["best_validation_auroc"]
)

assert np.isclose(
    saved_history["auroc"].max(),
    selected_row["auroc"]
)

# Recalculate metrics from saved predictions
recomputed = calculate_image_metrics(
    saved_images["label"],
    saved_images["probability"]
)

for metric in [
    "auroc", "auprc",
    "balanced_accuracy", "ece_10",
    "brier", "nll"
]:
    assert np.isclose(
        recomputed[metric],
        float(selected_row[metric]),
        atol=1e-6
    ), f"Mismatch in {metric}"

# Inspect checkpoint
saved_checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

assert saved_checkpoint["epoch"] == 5
assert saved_checkpoint["validation_fold"] == 1

# Calculate checkpoint hash for backup verification
sha256 = hashlib.sha256()

with open(checkpoint_path, "rb") as f:
    for chunk in iter(
        lambda: f.read(1024 * 1024),
        b""
    ):
        sha256.update(chunk)

checkpoint_hash = sha256.hexdigest()

print("\nBest epoch:", saved_summary["best_epoch"])
print("Best AUROC:", recomputed["auroc"])
print("Checkpoint SHA256:", checkpoint_hash)

print("\nPASS: Fold 1 artifacts verified.")


training_history.csv | 0.001 MB
best_validation_patch_predictions.csv | 0.296 MB
best_validation_image_predictions.csv | 0.036 MB
best_validation_summary.json | 0.001 MB
resnet18_fold1_best.pt | 42.715 MB

Best epoch: 5
Best AUROC: 0.9924934400637725
Checkpoint SHA256: 3aa81e0623d92f05f3fa0b484aca5235ed80122ba79a4e79aabb6b93d06ee559

PASS: Fold 1 artifacts verified.



## Formal ResNet-18 — Fold 1 Results

### Experimental Setup

The first formal experiment used an ImageNet-pretrained ResNet-18 with partial fine-tuning of `layer4` and the final classification layer.

The model was trained using 32 Development patients and validated using 17 separate Development patients. No Calibration or Test patients were used.

Training lasted five epochs, with the best checkpoint selected according to image-level AUROC.

### Results

| Metric | Selected Epoch 5 |
|---|---:|
| AUROC | 0.992493 |
| AUPRC | 0.996657 |
| Balanced Accuracy | 0.940462 |
| ECE (10 bins) | 0.074696 |
| Brier Score | 0.063208 |
| Negative Log-Likelihood | 0.256791 |

### Preliminary Interpretation

The model achieved strong discrimination on Development Fold 1. However, calibration performance varied considerably between epochs.

Epoch 2 had a slightly lower AUROC (0.991962) but better calibration metrics, including ECE (0.021038), Brier score (0.026373), and NLL (0.101626).

Epoch 3 showed an even larger calibration deterioration despite retaining high discrimination.

These observations support the importance of evaluating discrimination and calibration separately. However, they are preliminary Development-validation findings, not final evidence of external transportability.

Epoch 5 remains the selected checkpoint because image-level AUROC was the predefined selection criterion.

### Reproducibility

The selected Epoch 5 model checkpoint, patch-level predictions, image-level predictions, training history, and experimental configuration were saved.

Model checkpoint SHA-256:

`3aa81e0623d92f05f3fa0b484aca5235ed80122ba79a4e79aabb6b93d06ee559`

The model checkpoint is backed up separately from the GitHub repository.


In [ ]:

# ============================================================
# Back up Fold 1 model checkpoint
# ============================================================

from google.colab import files

assert checkpoint_path.exists()

print("Downloading:", checkpoint_path.name)
print("SHA256:", checkpoint_hash)

files.download(str(checkpoint_path))


Downloading: resnet18_fold1_best.pt
SHA256: 3aa81e0623d92f05f3fa0b484aca5235ed80122ba79a4e79aabb6b93d06ee559


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


## Formal ResNet-18 — Development Fold 2

### Objective

This experiment evaluates the fine-tuned ResNet-18 on the second patient-grouped Development validation fold.

### Data Partition

- Training: Development Folds 1 and 3
- Validation: Development Fold 2
- Training patients: 33
- Validation patients: 16
- Training images: 702
- Validation images: 431
- Training patches: 11,085
- Validation patches: 6,810

### Experimental Controls

The ResNet-18 architecture, pretrained weights, fine-tuning layers, optimizer, learning rates, weight decay, batch size, number of epochs, and preprocessing are unchanged from Fold 1.

A fresh model is initialized for Fold 2. No weights are transferred from the Fold 1 trained model.

Class weighting is calculated using Fold 2 training data only.

### Model Selection and Evaluation

The model is trained for five epochs. After each epoch, patch-level logits are aggregated into source-image predictions using the mean-logit method.

Image-level AUROC is the predefined checkpoint-selection metric. AUPRC, balanced accuracy, ECE, Brier score, and NLL are also recorded.

Calibration, Test, BACH, and BRACS data remain excluded.


In [ ]:

# ============================================================
# Prepare formal Fold 2 datasets and DataLoaders
# ============================================================

import torch
import numpy as np
import random

from torch.utils.data import DataLoader

FOLD = 2
FOLD_SEED = SEED + FOLD

BATCH_SIZE = TRAIN_CONFIG["batch_size"]

# Reuse workers throughout training to reduce
# repeated multiprocessing startup/cleanup.
NUM_WORKERS_FORMAL = 2

fold_train_df = all_patches.loc[
    all_patches["development_fold"] != FOLD
].copy()

fold_val_df = all_patches.loc[
    all_patches["development_fold"] == FOLD
].copy()

train_patients = set(fold_train_df["patient_id"])
val_patients = set(fold_val_df["patient_id"])

# Verify frozen Fold 2 assignments
assert train_patients.isdisjoint(val_patients)

assert train_patients | val_patients == set(
    all_patches["patient_id"]
)

assert len(train_patients) == 33
assert len(val_patients) == 16

assert fold_train_df["filename"].nunique() == 702
assert fold_val_df["filename"].nunique() == 431

assert len(fold_train_df) == 11085
assert len(fold_val_df) == 6810

assert set(fold_val_df["development_fold"]) == {2}
assert set(fold_train_df["development_fold"]) == {1, 3}


def seed_formal_worker(worker_id):
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


train_dataset = BreakHisPatchDataset(
    manifest=fold_train_df,
    dataset_root=DATASET_ROOT,
    transform=train_transform
)

val_dataset = BreakHisPatchDataset(
    manifest=fold_val_df,
    dataset_root=DATASET_ROOT,
    transform=val_transform
)

train_generator = torch.Generator()
train_generator.manual_seed(FOLD_SEED)

val_generator = torch.Generator()
val_generator.manual_seed(FOLD_SEED + 1000)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS_FORMAL,
    persistent_workers=(NUM_WORKERS_FORMAL > 0),
    pin_memory=True,
    worker_init_fn=seed_formal_worker,
    generator=train_generator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS_FORMAL,
    persistent_workers=(NUM_WORKERS_FORMAL > 0),
    pin_memory=True,
    worker_init_fn=seed_formal_worker,
    generator=val_generator
)

print("FORMAL RESNET-18 FOLD 2")
print("=" * 50)

print("Training patients:", len(train_patients))
print("Validation patients:", len(val_patients))

print("Training images:", fold_train_df["filename"].nunique())
print("Validation images:", fold_val_df["filename"].nunique())

print("Training patches:", len(train_dataset))
print("Validation patches:", len(val_dataset))

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

print("\nPASS: Fold 2 DataLoaders ready.")


FORMAL RESNET-18 FOLD 2
Training patients: 33
Validation patients: 16
Training images: 702
Validation images: 431
Training patches: 11085
Validation patches: 6810
Training batches: 347
Validation batches: 213

PASS: Fold 2 DataLoaders ready.


In [ ]:

# ============================================================
# Reusable formal ResNet-18 fold training
# ============================================================

import gc
import json

import torch
import torch.nn as nn

from pathlib import Path


def run_resnet18_fold(
    fold,
    train_df,
    val_df,
    train_loader,
    val_loader
):

    fold_seed = SEED + fold

    set_all_seeds(fold_seed)

    fold_results_dir = (
        RESULTS_DIR /
        "resnet18" /
        f"fold_{fold}"
    )

    fold_results_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    checkpoint_dir = Path(
        "/content/cis631_model_checkpoints"
    )

    checkpoint_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    checkpoint_file = (
        checkpoint_dir /
        f"resnet18_fold{fold}_best.pt"
    )

    history_file = (
        fold_results_dir /
        "training_history.csv"
    )

    patch_file = (
        fold_results_dir /
        "best_validation_patch_predictions.csv"
    )

    image_file = (
        fold_results_dir /
        "best_validation_image_predictions.csv"
    )

    summary_file = (
        fold_results_dir /
        "best_validation_summary.json"
    )

    # Prevent an accidental rerun from overwriting
    # a previously saved experiment.
    existing = [
        p for p in [
            checkpoint_file,
            history_file,
            patch_file,
            image_file,
            summary_file
        ]
        if p.exists()
    ]

    if existing:
        raise FileExistsError(
            "Fold outputs already exist. "
            "Back up and review them before rerunning:\n"
            + "\n".join(str(p) for p in existing)
        )

    # Check fold separation
    training_patients = set(train_df["patient_id"])
    validation_patients = set(val_df["patient_id"])

    assert training_patients.isdisjoint(
        validation_patients
    )

    assert validation_patients == set(
        development_folds.loc[
            development_folds["development_fold"] == fold,
            "patient_id"
        ]
    )

    # Initialize a fresh model for this fold
    model = build_resnet18_finetuned(
        seed=fold_seed
    ).to(device)

    # Class weighting from training patches only
    labels = train_df["label"].astype(int)

    n_benign = int((labels == 0).sum())
    n_malignant = int((labels == 1).sum())

    assert n_benign > 0
    assert n_malignant > 0

    pos_weight_value = (
        n_benign / n_malignant
    )

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(
            [pos_weight_value],
            dtype=torch.float32,
            device=device
        )
    )

    optimizer = torch.optim.AdamW(
        [
            {
                "params": model.layer4.parameters(),
                "lr": TRAIN_CONFIG["lr_layer4"]
            },
            {
                "params": model.fc.parameters(),
                "lr": TRAIN_CONFIG["lr_fc"]
            }
        ],
        weight_decay=TRAIN_CONFIG["weight_decay"]
    )

    scaler = torch.amp.GradScaler("cuda")

    best_auc = -float("inf")
    best_epoch = None
    best_metrics = None

    history_rows = []

    print("\n" + "=" * 60)
    print(f"RESNET-18 — FORMAL DEVELOPMENT FOLD {fold}")
    print("=" * 60)

    print("Training patients:", len(training_patients))
    print("Validation patients:", len(validation_patients))
    print("Training patches:", len(train_df))
    print("Validation patches:", len(val_df))
    print("Positive class weight:", round(pos_weight_value, 6))
    print("Epochs:", TRAIN_CONFIG["epochs"])
    print("=" * 60)

    for epoch in range(
        1,
        TRAIN_CONFIG["epochs"] + 1
    ):

        print(
            f"\nFold {fold} — "
            f"Epoch {epoch}/{TRAIN_CONFIG['epochs']}"
        )

        training_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            scaler=scaler,
            device=device
        )

        (
            validation_metrics,
            patch_predictions,
            image_predictions
        ) = validate_one_epoch(
            model=model,
            loader=val_loader,
            device=device
        )

        # Verify evaluation population
        assert len(patch_predictions) == len(val_df)

        assert len(image_predictions) == (
            val_df["filename"].nunique()
        )

        assert (
            image_predictions["patient_id"].nunique()
            == len(validation_patients)
        )

        assert set(
            image_predictions["patient_id"]
        ) == validation_patients

        epoch_result = {
            "epoch": epoch,
            "training_loss": training_loss,
            **validation_metrics
        }

        history_rows.append(epoch_result)

        print(f"Train loss: {training_loss:.5f}")
        print(f"AUROC:      {validation_metrics['auroc']:.6f}")
        print(f"AUPRC:      {validation_metrics['auprc']:.6f}")
        print(f"Balanced:   {validation_metrics['balanced_accuracy']:.6f}")
        print(f"ECE:        {validation_metrics['ece_10']:.6f}")
        print(f"Brier:      {validation_metrics['brier']:.6f}")
        print(f"NLL:        {validation_metrics['nll']:.6f}")

        # Select checkpoint by the predefined AUROC rule
        if validation_metrics["auroc"] > best_auc:

            best_auc = float(
                validation_metrics["auroc"]
            )

            best_epoch = epoch
            best_metrics = epoch_result.copy()

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "epoch": epoch,
                    "validation_fold": fold,
                    "seed": fold_seed,
                    "training_config": TRAIN_CONFIG,
                    "validation_metrics": validation_metrics
                },
                checkpoint_file
            )

            patch_predictions.to_csv(
                patch_file,
                index=False
            )

            image_predictions.to_csv(
                image_file,
                index=False
            )

            print(
                f"NEW BEST CHECKPOINT: Epoch {epoch}"
            )

        # Save running history after every epoch
        pd.DataFrame(
            history_rows
        ).to_csv(
            history_file,
            index=False
        )

    summary = {
        "architecture": "resnet18",
        "validation_fold": fold,
        "best_epoch": best_epoch,
        "best_validation_auroc": best_auc,
        "selection_metric": "image_auroc",
        "training_patients": len(training_patients),
        "validation_patients": len(validation_patients),
        "training_images": int(
            train_df["filename"].nunique()
        ),
        "validation_images": int(
            val_df["filename"].nunique()
        ),
        "training_patches": len(train_df),
        "validation_patches": len(val_df),
        "positive_class_weight": float(pos_weight_value),
        "checkpoint_path": str(checkpoint_file),
        "training_config": TRAIN_CONFIG,
        "best_epoch_metrics": best_metrics
    }

    with open(summary_file, "w") as f:
        json.dump(summary, f, indent=2)

    del model, optimizer, criterion, scaler
    gc.collect()
    torch.cuda.empty_cache()

    print("\n" + "=" * 60)
    print(f"FOLD {fold} COMPLETE")
    print("=" * 60)
    print("Best epoch:", best_epoch)
    print("Best AUROC:", round(best_auc, 6))

    return {
        "summary": summary,
        "history_path": history_file,
        "patch_path": patch_file,
        "image_path": image_file,
        "summary_path": summary_file,
        "checkpoint_path": checkpoint_file
    }


print("PASS: Reusable ResNet-18 training function defined.")


PASS: Reusable ResNet-18 training function defined.


In [ ]:

# ============================================================
# Train formal ResNet-18 Fold 2
# ============================================================

fold2_results = run_resnet18_fold(
    fold=2,
    train_df=fold_train_df,
    val_df=fold_val_df,
    train_loader=train_loader,
    val_loader=val_loader
)

print("\nPASS: Formal ResNet-18 Fold 2 training complete.")



RESNET-18 — FORMAL DEVELOPMENT FOLD 2
Training patients: 33
Validation patients: 16
Training patches: 11085
Validation patches: 6810
Positive class weight: 0.389794
Epochs: 5

Fold 2 — Epoch 1/5


Training:   0%|          | 0/347 [00:00<?, ?it/s]

Validation:   0%|          | 0/213 [00:00<?, ?it/s]

Train loss: 0.05542
AUROC:      0.941691
AUPRC:      0.976552
Balanced:   0.849839
ECE:        0.082834
Brier:      0.099506
NLL:        0.394914
NEW BEST CHECKPOINT: Epoch 1

Fold 2 — Epoch 2/5


Training:   0%|          | 0/347 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/213 [00:00<?, ?it/s]

Train loss: 0.00335
AUROC:      0.956628
AUPRC:      0.982234
Balanced:   0.842383
ECE:        0.099174
Brier:      0.103134
NLL:        0.532513
NEW BEST CHECKPOINT: Epoch 4

Fold 2 — Epoch 5/5


Training:   0%|          | 0/347 [00:00<?, ?it/s]

Validation:   0%|          | 0/213 [00:00<?, ?it/s]

Train loss: 0.00712
AUROC:      0.956628
AUPRC:      0.982722
Balanced:   0.874943
ECE:        0.083344
Brier:      0.091100
NLL:        0.475265

FOLD 2 COMPLETE
Best epoch: 4
Best AUROC: 0.956628

PASS: Formal ResNet-18 Fold 2 training complete.


In [ ]:

# ============================================================
# Verify Fold 2 training outputs
# ============================================================

import hashlib

history_df = pd.read_csv(
    fold2_results["history_path"]
)

image_df = pd.read_csv(
    fold2_results["image_path"],
    dtype={"patient_id": str}
)

patch_df = pd.read_csv(
    fold2_results["patch_path"],
    dtype={"patient_id": str}
)

summary = fold2_results["summary"]

assert len(history_df) == 5
assert len(image_df) == 431
assert len(patch_df) == 6810
assert image_df["patient_id"].nunique() == 16

assert summary["validation_fold"] == 2

# Verify selected epoch and AUROC
best_row = history_df.loc[
    history_df["epoch"] == summary["best_epoch"]
].iloc[0]

assert np.isclose(
    best_row["auroc"],
    history_df["auroc"].max()
)

assert np.isclose(
    best_row["auroc"],
    summary["best_validation_auroc"]
)

# Recompute all metrics from saved predictions
recomputed = calculate_image_metrics(
    image_df["label"],
    image_df["probability"]
)

for metric in [
    "auroc", "auprc",
    "balanced_accuracy", "ece_10",
    "brier", "nll"
]:
    assert np.isclose(
        recomputed[metric],
        best_row[metric],
        atol=1e-6
    ), f"Metric mismatch: {metric}"

checkpoint_path_fold2 = (
    fold2_results["checkpoint_path"]
)

assert checkpoint_path_fold2.is_file()

checkpoint = torch.load(
    checkpoint_path_fold2,
    map_location="cpu",
    weights_only=True
)

assert checkpoint["epoch"] == summary["best_epoch"]
assert checkpoint["validation_fold"] == 2

# Calculate checkpoint hash
hasher = hashlib.sha256()

with open(checkpoint_path_fold2, "rb") as f:
    for chunk in iter(
        lambda: f.read(1024 * 1024),
        b""
    ):
        hasher.update(chunk)

fold2_sha256 = hasher.hexdigest()

print("FOLD 2 EPOCH RESULTS\n")

display(
    history_df[
        [
            "epoch",
            "training_loss",
            "auroc",
            "auprc",
            "balanced_accuracy",
            "ece_10",
            "brier",
            "nll"
        ]
    ]
)

print("\nSelected epoch:", summary["best_epoch"])
print("Selected AUROC:", summary["best_validation_auroc"])

print("\nCheckpoint size (MB):",
      round(checkpoint_path_fold2.stat().st_size / 1024**2, 2))

print("Checkpoint SHA256:", fold2_sha256)

print("\nPASS: Fold 2 artifacts verified.")


FOLD 2 EPOCH RESULTS



,epoch,training_loss,auroc,auprc,balanced_accuracy,ece_10,brier,nll
0,1,0.055417,0.941691,0.976552,0.849839,0.082834,0.099506,0.394914
1,2,0.015123,0.947368,0.979033,0.837349,0.098105,0.104970,0.463433
2,3,0.013397,0.955568,0.981902,0.850709,0.094839,0.101543,0.388965
3,4,0.003349,0.956628,0.982234,0.842383,0.099174,0.103134,0.532513
4,5,0.007116,0.956628,0.982722,0.874943,0.083344,0.091100,0.475265



Selected epoch: 4
Selected AUROC: 0.9566281475500833

Checkpoint size (MB): 42.71
Checkpoint SHA256: 7cac9c1b304d00e58798e3dbe5a65170e7c56780b998961714dc4c18158e15bd

PASS: Fold 2 artifacts verified.


In [ ]:

# ============================================================
# Inspect Fold 2 AUROC selection precision
# ============================================================

history = pd.read_csv(
    fold2_results["history_path"]
)

epoch4_auc = float(
    history.loc[
        history["epoch"] == 4,
        "auroc"
    ].iloc[0]
)

epoch5_auc = float(
    history.loc[
        history["epoch"] == 5,
        "auroc"
    ].iloc[0]
)

print(f"Epoch 4 AUROC: {epoch4_auc:.12f}")
print(f"Epoch 5 AUROC: {epoch5_auc:.12f}")

print(
    f"Difference (Epoch 4 - Epoch 5): "
    f"{epoch4_auc - epoch5_auc:.12f}"
)

selected_epoch = int(
    history.loc[
        history["auroc"].idxmax(),
        "epoch"
    ]
)

assert selected_epoch == 4
assert fold2_results["summary"]["best_epoch"] == 4

print("\nPASS: Fold 2 checkpoint selection verified.")


Epoch 4 AUROC: 0.956628147550
Epoch 5 AUROC: 0.956628147550
Difference (Epoch 4 - Epoch 5): 0.000000000000

PASS: Fold 2 checkpoint selection verified.



## Formal ResNet-18 — Fold 2 Results and Interpretation

### Experimental Setup

The second formal experiment used the same partially fine-tuned, ImageNet-pretrained ResNet-18 configuration as Fold 1.

- Training: Development Folds 1 and 3
- Validation: Development Fold 2
- Training patients: 33
- Validation patients: 16
- Training images: 702
- Validation images: 431
- Training patches: 11,085
- Validation patches: 6,810
- Training-only positive class weight: 0.389794
- Training epochs: 5

The model was initialized from ImageNet-pretrained weights independently of Fold 1. BatchNorm running statistics remained frozen.

### Selected Checkpoint

Epoch 4 was selected using the predefined image-level AUROC criterion.

| Metric | Selected Epoch 4 |
|---|---:|
| AUROC | 0.956628 |
| AUPRC | 0.982234 |
| Balanced Accuracy | 0.842383 |
| ECE (10 bins) | 0.099174 |
| Brier Score | 0.103134 |
| Negative Log-Likelihood | 0.532513 |

### Preliminary Interpretation

The Fold 2 model achieved strong discrimination, although its performance was lower than the previously completed Fold 1 experiment.

The validation results also showed differences between discrimination and calibration across epochs. In particular, Epoch 5 had better ECE, Brier score, NLL, and balanced accuracy than Epoch 4, despite their nearly identical displayed AUROC values.

Epoch 4 remains the selected checkpoint because AUROC was the predefined model-selection criterion.

These results suggest that model selection based on discrimination alone does not necessarily identify the checkpoint with the most reliable probability predictions.

However, these observations are based on Development validation only. They do not establish the final model's calibration or transportability under distribution shift.

### Reproducibility

The training history, selected validation predictions, model configuration, and checkpoint metadata have been preserved.

Checkpoint SHA-256:

`7cac9c1b304d00e58798e3dbe5a65170e7c56780b998961714dc4c18158e15bd`

The selected model checkpoint is backed up separately from the GitHub repository.

